In [1]:
!pip install fastapi uvicorn httpx pyngrok
!pip install faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 55.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 50.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 74.5 MB/s eta 0:00:00:00:0100:01


In [2]:
from pyngrok import ngrok
ngrok.set_auth_token("2xZqpiP0G671IWuo5QikyhoWuYx_5CqC4z66tMzYaTcctUMzC")

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
REGISTRY_SECRET = user_secrets.get_secret("REGISTRY_SECRET")
REGISTRY_URL = user_secrets.get_secret("REGISTRY_URL")
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

In [ ]:
import os, uuid, asyncio, threading, time
import torch
import httpx
import torchaudio
import nest_asyncio
from fastapi import FastAPI, File, UploadFile, Header, HTTPException
import uvicorn
from faster_whisper import WhisperModel

# 🔧 allow uvicorn inside notebook
nest_asyncio.apply()

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME = "whisper-large-v3"
SESSION_ID = str(uuid.uuid4())

# ✅ Prevent silent crashes later
if not REGISTRY_URL:
    raise RuntimeError("REGISTRY_URL is required")

# Load model once
model = WhisperModel("large-v3", device="cuda", compute_type="float16")

app = FastAPI()

# ── API ───────────────────────────────────────────────────────────────────────
@app.post("/transcribe_audio")
async def transcribe_audio(
    audio: UploadFile = File(...),
    x_registry_token: str = Header(None)
):
    # Optional auth (only if secret exists)
    if REGISTRY_SECRET and x_registry_token != REGISTRY_SECRET:
        raise HTTPException(status_code=403, detail="Forbidden")

    audio_bytes = await audio.read()

    file_path = f"/tmp/{uuid.uuid4()}.mp3"
    with open(file_path, "wb") as f:
        f.write(audio_bytes)

    segments, info = await asyncio.to_thread(
        model.transcribe, file_path, language="ur"
    )

    transcript = " ".join([s.text for s in segments])

    return {"transcript": transcript}


# ── Registry Loop ─────────────────────────────────────────────────────────────
def registry_loop(public_url: str):
    # ✅ NEVER allow None in headers
    headers = {
        k: v for k, v in {
            "X-Registry-Token": REGISTRY_SECRET
        }.items() if v is not None
    }

    with httpx.Client(timeout=10.0) as client:
        # Register once
        try:
            client.post(
                f"{REGISTRY_URL}/register",
                json={
                    "name": MODEL_NAME,
                    "endpoint": f"{public_url}/transcribe_audio",
                    "session_id": SESSION_ID,
                },
                headers=headers
            )
            print("✓ Registered with registry")
        except Exception as e:
            print(f"Register failed: {e}")

        # Keep pinging
        while True:
            try:
                client.post(
                    f"{REGISTRY_URL}/ping",
                    json={
                        "name": MODEL_NAME,
                        "session_id": SESSION_ID,
                    },
                    headers=headers
                )
            except Exception as e:
                print(f"Ping failed: {e}")

            time.sleep(5)


# ── Main ──────────────────────────────────────────────────────────────────────
async def main():
    # Start ngrok ONCE
    tunnel = ngrok.connect(8000)
    public_url = tunnel.public_url

    print(f"ngrok URL: {public_url}")

    # Start registry thread ONCE
    threading.Thread(
        target=registry_loop,
        args=(public_url,),
        daemon=True
    ).start()

    # Start server
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    await server.serve()


# Run
await main()

INFO:     Started server process [55]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


ngrok URL: https://ae4f-136-115-5-155.ngrok-free.app
✓ Registered with registry
